In [1]:
#importing data and checking shape
#498,094 providers and 12 model features + NPI + the 2 anomaly columns
import pandas as pd

anomaly_results = pd.read_parquet(
    "../data/processed/provider_anomaly_results.parquet"
)

print(anomaly_results.shape)
anomaly_results.head()

(498094, 15)


,provider_npi,log_total_claim_lines,log_unique_claims,log_unique_beneficiaries,log_unique_hcpcs,log_total_payment,avg_payment,active_months,log_avg_monthly_claim_lines,log_max_monthly_claim_lines,log_avg_monthly_payment,std_monthly_claim_lines,std_monthly_payment,anomaly_prediction,anomaly_score
0,102408,1.386294,1.098612,1.098612,1.098612,4.948760,46.666667,2,0.916291,1.098612,4.262680,0.707107,14.142136,1,0.215481
1,114194,0.693147,0.693147,0.693147,0.693147,3.433987,30.000000,1,0.693147,0.693147,3.433987,0.000000,0.000000,1,0.230298
2,143410,1.098612,0.693147,0.693147,0.693147,3.433987,15.000000,1,1.098612,1.098612,3.433987,0.000000,0.000000,1,0.198438
3,145002,2.833213,1.945910,1.945910,2.772589,7.170888,81.250000,6,1.299283,1.791759,5.382965,1.966384,458.461194,1,0.098051
4,186786,1.098612,0.693147,0.693147,0.693147,3.433987,15.000000,1,1.098612,1.098612,3.433987,0.000000,0.000000,1,0.198438


In [2]:
#most anamolous providers: 
anomaly_results.sort_values(
    "anomaly_score"
).head(10)[
    ["provider_npi", "anomaly_prediction", "anomaly_score"]
]

,provider_npi,anomaly_prediction,anomaly_score
497551,9979265126,-1,-0.170108
177303,7132269719,-1,-0.170108
297860,1973668724,-1,-0.170108
197996,7961400834,-1,-0.170108
374593,5040521237,-1,-0.170108
413117,6585316279,-1,-0.170108
337345,3552796390,-1,-0.170108
22139,888901330,-1,-0.170108
272288,941734431,-1,-0.170108
257072,329757128,-1,-0.170108


In [3]:
#function to get provider profile by NPI (tool 1)
def get_provider_profile(provider_npi):
    provider = anomaly_results[
        anomaly_results["provider_npi"].astype(str).str.strip()
        == str(provider_npi).strip()
    ]

    if provider.empty:
        return {"error": f"Provider {provider_npi} not found"}

    row = provider.iloc[0]

    return {
        "provider_npi": str(row["provider_npi"]),
        "anomaly_prediction": int(row["anomaly_prediction"]),
        "anomaly_score": float(row["anomaly_score"]),
        "log_total_claim_lines": float(row["log_total_claim_lines"]),
        "log_unique_claims": float(row["log_unique_claims"]),
        "log_unique_beneficiaries": float(row["log_unique_beneficiaries"]),
        "log_unique_hcpcs": float(row["log_unique_hcpcs"]),
        "log_total_payment": float(row["log_total_payment"]),
        "avg_payment": float(row["avg_payment"]),
        "active_months": float(row["active_months"]),
        "log_avg_monthly_claim_lines": float(row["log_avg_monthly_claim_lines"]),
        "log_max_monthly_claim_lines": float(row["log_max_monthly_claim_lines"]),
        "log_avg_monthly_payment": float(row["log_avg_monthly_payment"]),
        "std_monthly_claim_lines": float(row["std_monthly_claim_lines"]),
        "std_monthly_payment": float(row["std_monthly_payment"])
    }

In [4]:
print(anomaly_results["provider_npi"].dtype)
print(anomaly_results["provider_npi"].head())

print(anomaly_results[
    anomaly_results["provider_npi"].astype(str).str.strip() == "9979265126"
])

int64
0    102408
1    114194
2    143410
3    145002
4    186786
Name: provider_npi, dtype: int64
        provider_npi  log_total_claim_lines  log_unique_claims  \
497551    9979265126               9.809342           9.187993   

        log_unique_beneficiaries  log_unique_hcpcs  log_total_payment  \
497551                  8.796188          7.361375          13.614228   

        avg_payment  active_months  log_avg_monthly_claim_lines  \
497551    44.922536             36                     6.227744   

        log_max_monthly_claim_lines  log_avg_monthly_payment  \
497551                     6.552508                10.030751   

        std_monthly_claim_lines  std_monthly_payment  anomaly_prediction  \
497551               151.544482          7156.817329                  -1   

        anomaly_score  
497551      -0.170108  


In [5]:
#initialize Spark session
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HealthcareInvestigationAgents")
    .getOrCreate()
)

print("Spark initialized.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/05 08:51:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark initialized.


In [6]:
#load data
provider_time_features = spark.read.parquet(
    "../data/processed/provider_time_features"
)

provider_time_features.printSchema()

#check columns: 
provider_time_features.columns

root
 |-- provider_npi: long (nullable = true)
 |-- active_months: long (nullable = true)
 |-- avg_monthly_claim_lines: double (nullable = true)
 |-- max_monthly_claim_lines: long (nullable = true)
 |-- std_monthly_claim_lines: double (nullable = true)
 |-- avg_monthly_payment: double (nullable = true)
 |-- max_monthly_payment: double (nullable = true)
 |-- std_monthly_payment: double (nullable = true)
 |-- avg_monthly_unique_beneficiaries: double (nullable = true)



['provider_npi',
 'active_months',
 'avg_monthly_claim_lines',
 'max_monthly_claim_lines',
 'std_monthly_claim_lines',
 'avg_monthly_payment',
 'max_monthly_payment',
 'std_monthly_payment',
 'avg_monthly_unique_beneficiaries']

In [7]:
#tool 2: retrieve provider's monthly activity history
def get_provider_monthly_activity(provider_npi):
    """
    Retrieve month-by-month claims activity for a provider.
    """

    try:
        provider_npi = int(provider_npi)
    except (TypeError, ValueError):
        return {"error": f"Invalid provider NPI: {provider_npi}"}

    monthly_df = (
        monthly_provider_activity
        .filter(F.col("provider_npi") == provider_npi)
        .orderBy("claim_month")
    )

    rows = monthly_df.collect()

    if not rows:
        return {
            "error": f"No monthly activity found for provider {provider_npi}"
        }

    return [row.asDict() for row in rows]

In [8]:
print("monthly_provider_activity exists:", "monthly_provider_activity" in globals())
print("spark exists:", "spark" in globals())

monthly_provider_activity exists: False
spark exists: True


In [9]:
monthly_provider_activity = spark.read.parquet(
    "../data/processed/monthly_provider_activity"
)
print("monthly_provider_activity exists:", "monthly_provider_activity" in globals())
monthly_provider_activity.printSchema()

monthly_provider_activity exists: True
root
 |-- provider_npi: long (nullable = true)
 |-- claim_month: string (nullable = true)
 |-- monthly_claim_lines: long (nullable = true)
 |-- monthly_unique_claims: long (nullable = true)
 |-- monthly_unique_beneficiaries: long (nullable = true)
 |-- monthly_payment: double (nullable = true)
 |-- monthly_unique_hcpcs: long (nullable = true)



In [10]:
from pyspark.sql import functions as F

In [11]:
#test the function with a sample provider
monthly_profile = get_provider_monthly_activity("9979265126")

monthly_profile

[{'provider_npi': 9979265126,
  'claim_month': '2008-01',
  'monthly_claim_lines': 200,
  'monthly_unique_claims': 127,
  'monthly_unique_beneficiaries': 126,
  'monthly_payment': 9150.0,
  'monthly_unique_hcpcs': 103},
 {'provider_npi': 9979265126,
  'claim_month': '2008-02',
  'monthly_claim_lines': 373,
  'monthly_unique_claims': 216,
  'monthly_unique_beneficiaries': 210,
  'monthly_payment': 16280.0,
  'monthly_unique_hcpcs': 171},
 {'provider_npi': 9979265126,
  'claim_month': '2008-03',
  'monthly_claim_lines': 457,
  'monthly_unique_claims': 258,
  'monthly_unique_beneficiaries': 248,
  'monthly_payment': 19300.0,
  'monthly_unique_hcpcs': 202},
 {'provider_npi': 9979265126,
  'claim_month': '2008-04',
  'monthly_claim_lines': 601,
  'monthly_unique_claims': 333,
  'monthly_unique_beneficiaries': 327,
  'monthly_payment': 28100.0,
  'monthly_unique_hcpcs': 252},
 {'provider_npi': 9979265126,
  'claim_month': '2008-05',
  'monthly_claim_lines': 700,
  'monthly_unique_claims': 37

In [12]:
#define population reference points for anomaly detection (tool 3)
import numpy as np
import pandas as pd

# Features used by the anomaly detection model
comparison_features = [
    "log_total_claim_lines",
    "log_unique_claims",
    "log_unique_beneficiaries",
    "log_unique_hcpcs",
    "log_total_payment",
    "avg_payment",
    "active_months",
    "log_avg_monthly_claim_lines",
    "log_max_monthly_claim_lines",
    "log_avg_monthly_payment",
    "std_monthly_claim_lines",
    "std_monthly_payment"
]

# Precompute population reference points once
population_reference = {}

for feature in comparison_features:
    population_reference[feature] = {
        "median": anomaly_results[feature].median(),
        "p90": anomaly_results[feature].quantile(0.90),
        "p95": anomaly_results[feature].quantile(0.95),
        "p99": anomaly_results[feature].quantile(0.99),
        "p999": anomaly_results[feature].quantile(0.999)
    }

print("Population reference statistics calculated.")

Population reference statistics calculated.


In [13]:
#tool 3: compare provider's features to population reference points
def compare_provider_to_population(provider_npi):
    """
    Compare a provider's anomaly-detection features
    against the broader provider population.
    """

    try:
        provider_npi = int(provider_npi)
    except (TypeError, ValueError):
        return {"error": f"Invalid provider NPI: {provider_npi}"}

    provider_rows = anomaly_results[
        anomaly_results["provider_npi"] == provider_npi
    ]

    if provider_rows.empty:
        return {
            "error": f"No provider found for NPI {provider_npi}"
        }

    provider = provider_rows.iloc[0]

    comparisons = {}

    for feature in comparison_features:
        value = provider[feature]
        reference = population_reference[feature]

        comparisons[feature] = {
            "provider_value": float(value),
            "population_median": float(reference["median"]),
            "population_p90": float(reference["p90"]),
            "population_p95": float(reference["p95"]),
            "population_p99": float(reference["p99"]),
            "population_p999": float(reference["p999"])
        }

    return {
        "provider_npi": str(provider_npi),
        "comparisons": comparisons
    }

In [14]:
population_comparison = compare_provider_to_population("9979265126")

population_comparison

{'provider_npi': '9979265126',
 'comparisons': {'log_total_claim_lines': {'provider_value': 9.809341694645898,
   'population_median': 1.6094379124341003,
   'population_p90': 2.9444389791664403,
   'population_p95': 3.258096538021482,
   'population_p99': 3.912023005428146,
   'population_p999': 5.302841066698872},
  'log_unique_claims': {'provider_value': 9.187992508312275,
   'population_median': 1.0986122886681096,
   'population_p90': 2.302585092994046,
   'population_p95': 2.639057329615259,
   'population_p99': 3.258096538021482,
   'population_p999': 4.663439094112067},
  'log_unique_beneficiaries': {'provider_value': 8.796187635470453,
   'population_median': 1.0986122886681096,
   'population_p90': 2.1972245773362196,
   'population_p95': 2.4849066497880004,
   'population_p99': 3.044522437723423,
   'population_p999': 4.330733340286331},
  'log_unique_hcpcs': {'provider_value': 7.3613754289773485,
   'population_median': 1.6094379124341003,
   'population_p90': 2.77258872223

In [15]:
[name for name in dir() if "claim" in name.lower() or "hcpcs" in name.lower()]

[]

In [16]:
provider_hcpcs_summary = spark.read.parquet(
    "../data/processed/provider_hcpcs_summary"
)

provider_hcpcs_summary.printSchema()

root
 |-- provider_npi: long (nullable = true)
 |-- hcpcs_code: string (nullable = true)
 |-- claim_lines: long (nullable = true)
 |-- unique_claims: long (nullable = true)
 |-- unique_beneficiaries: long (nullable = true)
 |-- total_payment: double (nullable = true)



In [17]:
#tool 4: retrieve provider's HCPCS/service code activity
def get_provider_hcpcs_activity(provider_npi, top_n=10):
    """
    Retrieve the HCPCS/service codes driving a provider's activity.
    """

    try:
        provider_npi = int(provider_npi)
    except (TypeError, ValueError):
        return {"error": f"Invalid provider NPI: {provider_npi}"}

    provider_df = (
        provider_hcpcs_summary
        .filter(F.col("provider_npi") == provider_npi)
    )

    # Calculate provider-level totals
    totals = provider_df.agg(
        F.sum("claim_lines").alias("total_claim_lines"),
        F.sum("total_payment").alias("total_payment")
    ).collect()[0]

    total_claim_lines = totals["total_claim_lines"] or 0
    total_payment = totals["total_payment"] or 0

    if total_claim_lines == 0:
        return {
            "error": f"No HCPCS activity found for provider {provider_npi}"
        }

    # Get highest-payment HCPCS codes
    rows = (
        provider_df
        .orderBy(F.desc("total_payment"))
        .limit(top_n)
        .collect()
    )

    records = []

    for row in rows:
        record = row.asDict()

        record["payment_share"] = (
            record["total_payment"] / total_payment
            if total_payment > 0
            else 0
        )

        record["claim_line_share"] = (
            record["claim_lines"] / total_claim_lines
            if total_claim_lines > 0
            else 0
        )

        records.append(record)

    top_codes_payment = sum(
        record["total_payment"] or 0
        for record in records
    )

    return {
        "provider_npi": str(provider_npi),
        "total_claim_lines": total_claim_lines,
        "total_payment": total_payment,
        "top_hcpcs_count": len(records),
        "top_hcpcs_by_payment": records,
        "top_codes_payment": top_codes_payment,
        "top_codes_payment_share": (
            top_codes_payment / total_payment
            if total_payment > 0
            else 0
        )
    }

In [18]:
#test tool on anomaly provider
hcpcs_profile = get_provider_hcpcs_activity("9979265126")

hcpcs_profile

{'provider_npi': '9979265126',
 'total_claim_lines': 18202,
 'total_payment': 817680.0,
 'top_hcpcs_count': 10,
 'top_hcpcs_by_payment': [{'provider_npi': 9979265126,
   'hcpcs_code': '99213',
   'claim_lines': 1286,
   'unique_claims': 1281,
   'unique_beneficiaries': 1215,
   'total_payment': 71600.0,
   'payment_share': 0.08756481753253106,
   'claim_line_share': 0.07065157674980771},
  {'provider_npi': 9979265126,
   'hcpcs_code': '99214',
   'claim_lines': 852,
   'unique_claims': 842,
   'unique_beneficiaries': 810,
   'total_payment': 47430.0,
   'payment_share': 0.058005576753742294,
   'claim_line_share': 0.04680804307218987},
  {'provider_npi': 9979265126,
   'hcpcs_code': '99232',
   'claim_lines': 494,
   'unique_claims': 360,
   'unique_beneficiaries': 343,
   'total_payment': 31120.0,
   'payment_share': 0.038058898346541435,
   'claim_line_share': 0.027139874739039668},
  {'provider_npi': 9979265126,
   'hcpcs_code': '36415',
   'claim_lines': 723,
   'unique_claims': 62

In [34]:
#LangGraph investigation workflow
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class InvestigationState(TypedDict):
    provider_npi: str
    profile: dict
    monthly_activity: dict
    population_comparison: dict
    hcpcs_activity: dict
    monthly_trends: dict
    report: dict

def collect_evidence(state: InvestigationState):
    npi = state["provider_npi"]

    return {
        "profile": get_provider_profile(npi),
        "monthly_activity": get_provider_monthly_activity(npi),
        "population_comparison": compare_provider_to_population(npi),
        "hcpcs_activity": get_provider_hcpcs_activity(npi)
    }

graph_builder = StateGraph(InvestigationState)

graph_builder.add_node("collect_evidence", collect_evidence)

graph_builder.add_edge(START, "collect_evidence")
graph_builder.add_edge("collect_evidence", END)

investigation_graph = graph_builder.compile()

In [20]:
#test the function with a sample provider
result = investigation_graph.invoke({
    "provider_npi": "9979265126",
    "profile": {},
    "monthly_activity": {},
    "population_comparison": {},
    "hcpcs_activity": {}
})

result.keys()

dict_keys(['provider_npi', 'profile', 'monthly_activity', 'population_comparison', 'hcpcs_activity'])

In [21]:
#test each tool and print results for the sample provider
for name in [
    "monthly_activity",
    "population_comparison",
    "hcpcs_activity"
]:
    value = result[name]
    print(f"\n{name.upper()}")
    print("Type:", type(value).__name__)

    if isinstance(value, dict):
        print("Keys:", list(value.keys()))
        print("Preview:", str(value)[:1000])
    elif isinstance(value, list):
        print("Number of records:", len(value))
        print("First record:", value[0] if value else "Empty list")
    else:
        print("Value:", str(value)[:1000])


MONTHLY_ACTIVITY
Type: list
Number of records: 36
First record: {'provider_npi': 9979265126, 'claim_month': '2008-01', 'monthly_claim_lines': 200, 'monthly_unique_claims': 127, 'monthly_unique_beneficiaries': 126, 'monthly_payment': 9150.0, 'monthly_unique_hcpcs': 103}

POPULATION_COMPARISON
Type: dict
Keys: ['provider_npi', 'comparisons']
Preview: {'provider_npi': '9979265126', 'comparisons': {'log_total_claim_lines': {'provider_value': 9.809341694645898, 'population_median': 1.6094379124341003, 'population_p90': 2.9444389791664403, 'population_p95': 3.258096538021482, 'population_p99': 3.912023005428146, 'population_p999': 5.302841066698872}, 'log_unique_claims': {'provider_value': 9.187992508312275, 'population_median': 1.0986122886681096, 'population_p90': 2.302585092994046, 'population_p95': 2.639057329615259, 'population_p99': 3.258096538021482, 'population_p999': 4.663439094112067}, 'log_unique_beneficiaries': {'provider_value': 8.796187635470453, 'population_median': 1.0986122

In [22]:
#month trend analysis function ADDED
def analyze_monthly_trends(state: InvestigationState):
    monthly = state["monthly_activity"]

    if not monthly:
        return {
            "monthly_trends": {
                "error": "No monthly activity available"
            }
        }

    # Sort records chronologically
    monthly = sorted(
        monthly,
        key=lambda row: row["claim_month"]
    )

    # Find peak months
    highest_payment_month = max(
        monthly,
        key=lambda row: row["monthly_payment"] or 0
    )

    highest_claim_month = max(
        monthly,
        key=lambda row: row["monthly_claim_lines"] or 0
    )

    # Calculate month-over-month payment changes
    changes = []

    for i in range(1, len(monthly)):
        previous = monthly[i - 1]
        current = monthly[i]

        previous_payment = previous["monthly_payment"] or 0
        current_payment = current["monthly_payment"] or 0

        change = current_payment - previous_payment

        if previous_payment != 0:
            pct_change = change / previous_payment
        else:
            pct_change = None

        changes.append({
            "from_month": previous["claim_month"],
            "to_month": current["claim_month"],
            "payment_change": change,
            "payment_pct_change": pct_change
        })

    largest_increase = max(
        changes,
        key=lambda row: row["payment_change"]
    ) if changes else None

    largest_decrease = min(
        changes,
        key=lambda row: row["payment_change"]
    ) if changes else None

    total_payment = sum(
        row["monthly_payment"] or 0
        for row in monthly
    )

    total_claims = sum(
        row["monthly_claim_lines"] or 0
        for row in monthly
    )

    return {
        "monthly_trends": {
            "months_observed": len(monthly),
            "average_monthly_payment": total_payment / len(monthly),
            "average_monthly_claim_lines": total_claims / len(monthly),
            "highest_payment_month": highest_payment_month,
            "highest_claim_month": highest_claim_month,
            "largest_payment_increase": largest_increase,
            "largest_payment_decrease": largest_decrease
        }
    }

In [40]:
#create sturctured investigation report 
def build_investigation_report(state: InvestigationState):
    profile = state["profile"]
    monthly = state["monthly_activity"]
    comparison = state["population_comparison"]
    hcpcs = state["hcpcs_activity"]
    trends = state["monthly_trends"]
    consistency = state["monthly_consistency"]

    # Identify features above the population's 99th percentile
    elevated_features = []
    for feature, values in comparison.get("comparisons", {}).items():
        provider_value = values.get("provider_value")
        p99 = values.get("population_p99")

        if provider_value is not None and p99 is not None:
            if provider_value > p99:
                elevated_features.append({
                    "feature": feature,
                    "provider_value": provider_value,
                    "population_p99": p99
                })

    return {
        "report": {
            "provider_npi": state["provider_npi"],
            "screening": {
                "anomaly_prediction": profile.get("anomaly_prediction"),
                "anomaly_score": profile.get("anomaly_score"),
                "interpretation": (
                    "Flagged by the Isolation Forest for further review."
                    if profile.get("anomaly_prediction") == -1
                    else "Not flagged by the Isolation Forest."
                )
            },
            "population_context": {
                "features_above_p99": elevated_features
            },
            "activity_summary": {
                "months_observed": len(monthly),
                "total_claim_lines": hcpcs.get("total_claim_lines"),
                "total_payment": hcpcs.get("total_payment")
            },
            
            "monthly_trend_analysis": trends,

            "hcpcs_summary": {
                "top_codes_by_payment": hcpcs.get(
                    "top_hcpcs_by_payment", []
                ),
                "top_codes_payment_share": hcpcs.get(
                    "top_codes_payment_share"
                )
            },

            "monthly_consistency": consistency,

            "limitations": [
                "An anomaly flag is not evidence of fraud.",
                "Population comparisons are descriptive, not causal.",
                "Log-transformed features are reported in log space."
            ]
        }
    }

In [35]:
#graph includes report node
graph_builder = StateGraph(InvestigationState)

graph_builder.add_node("collect_evidence", collect_evidence)
graph_builder.add_node("analyze_trends", analyze_monthly_trends)
graph_builder.add_node(
    "analyze_consistency",
    lambda state: {
        "monthly_consistency": analyze_monthly_consistency(state)
    }
)
graph_builder.add_node("build_report", build_investigation_report)

graph_builder.add_edge(START, "collect_evidence")
graph_builder.add_edge("collect_evidence", "analyze_trends")
graph_builder.add_edge("analyze_trends", "analyze_consistency")
graph_builder.add_edge("analyze_consistency", "build_report")
graph_builder.add_edge("build_report", END)

investigation_graph = graph_builder.compile()

#TypedDict
class InvestigationState(TypedDict):
    provider_npi: str
    profile: dict
    monthly_activity: list
    population_comparison: dict
    hcpcs_activity: dict
    monthly_trends: dict
    monthly_consistency: dict
    report: dict



In [38]:
def analyze_consistency(state: InvestigationState):
    return {
        "monthly_consistency": analyze_monthly_consistency(state)
    }

In [26]:
print(investigation_graph.get_graph().nodes.keys())


dict_keys(['__start__', 'collect_evidence', 'analyze_trends', 'analyze_consistency', 'build_report', '__end__'])


In [27]:
test_state = {
    "provider_npi": "9979265126",
    "profile": result["profile"],
    "monthly_activity": result["monthly_activity"],
    "population_comparison": result["population_comparison"],
    "hcpcs_activity": result["hcpcs_activity"],
    "monthly_consistency": {},
    "monthly_trends": {}
}

test_report = build_investigation_report(test_state)

print(test_report.keys())
print(test_report["report"])

dict_keys(['report'])
{'provider_npi': '9979265126', 'screening': {'anomaly_prediction': -1, 'anomaly_score': -0.1701077072165179, 'interpretation': 'Flagged by the Isolation Forest for further review.'}, 'population_context': {'features_above_p99': [{'feature': 'log_total_claim_lines', 'provider_value': 9.809341694645898, 'population_p99': 3.912023005428146}, {'feature': 'log_unique_claims', 'provider_value': 9.187992508312275, 'population_p99': 3.258096538021482}, {'feature': 'log_unique_beneficiaries', 'provider_value': 8.796187635470453, 'population_p99': 3.044522437723423}, {'feature': 'log_unique_hcpcs', 'provider_value': 7.3613754289773485, 'population_p99': 3.6375861597263857}, {'feature': 'log_total_payment', 'provider_value': 13.614227563974522, 'population_p99': 7.779466967458324}, {'feature': 'active_months', 'provider_value': 36.0, 'population_p99': 18.0}, {'feature': 'log_avg_monthly_claim_lines', 'provider_value': 6.2277436700375, 'population_p99': 2.0794415416798357}, {

In [41]:
#test the function with a sample provider
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class InvestigationState(TypedDict):
    provider_npi: str
    profile: dict
    monthly_activity: list
    population_comparison: dict
    hcpcs_activity: dict
    monthly_trends: dict
    monthly_consistency: dict
    report: dict


graph_builder = StateGraph(InvestigationState)

graph_builder.add_node("collect_evidence", collect_evidence)
graph_builder.add_node("analyze_trends", analyze_monthly_trends)
graph_builder.add_node("analyze_consistency", analyze_consistency)
graph_builder.add_node("build_report", build_investigation_report)

graph_builder.add_edge(START, "collect_evidence")
graph_builder.add_edge("collect_evidence", "analyze_trends")
graph_builder.add_edge("analyze_trends", "analyze_consistency")
graph_builder.add_edge("analyze_consistency", "build_report")
graph_builder.add_edge("build_report", END)

investigation_graph = graph_builder.compile()

In [29]:
print(investigation_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	collect_evidence(collect_evidence)
	analyze_trends(analyze_trends)
	analyze_consistency(analyze_consistency)
	build_report(build_report)
	__end__([<p>__end__</p>]):::last
	__start__ --> collect_evidence;
	analyze_consistency --> build_report;
	analyze_trends --> analyze_consistency;
	collect_evidence --> analyze_trends;
	build_report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [39]:
print(analyze_consistency({
    "monthly_activity": result["monthly_activity"]
}))

{'monthly_consistency': {'above_average_months': 23, 'months_observed': 36, 'top_five_months': [{'provider_npi': 9979265126, 'claim_month': '2008-05', 'monthly_claim_lines': 700, 'monthly_unique_claims': 373, 'monthly_unique_beneficiaries': 365, 'monthly_payment': 28510.0, 'monthly_unique_hcpcs': 275}, {'provider_npi': 9979265126, 'claim_month': '2008-10', 'monthly_claim_lines': 672, 'monthly_unique_claims': 356, 'monthly_unique_beneficiaries': 350, 'monthly_payment': 27110.0, 'monthly_unique_hcpcs': 282}, {'provider_npi': 9979265126, 'claim_month': '2009-07', 'monthly_claim_lines': 672, 'monthly_unique_claims': 344, 'monthly_unique_beneficiaries': 341, 'monthly_payment': 29820.0, 'monthly_unique_hcpcs': 288}, {'provider_npi': 9979265126, 'claim_month': '2008-09', 'monthly_claim_lines': 666, 'monthly_unique_claims': 338, 'monthly_unique_beneficiaries': 334, 'monthly_payment': 29080.0, 'monthly_unique_hcpcs': 279}, {'provider_npi': 9979265126, 'claim_month': '2008-07', 'monthly_claim_li

In [42]:
#end-to-end LangGraph investigation pipeline that collects evidence from four tools and builds a structured report.
result = investigation_graph.invoke({
    "provider_npi": "9979265126",
    "profile": {},
    "monthly_activity": [],
    "population_comparison": {},
    "hcpcs_activity": {},
    "monthly_trends": {},
    "monthly_consistency": {},
    "report": {}
})

from pprint import pprint

pprint(result["report"]["monthly_trend_analysis"])

{'average_monthly_claim_lines': 505.6111111111111,
 'average_monthly_payment': 22713.333333333332,
 'highest_claim_month': {'claim_month': '2008-05',
                         'monthly_claim_lines': 700,
                         'monthly_payment': 28510.0,
                         'monthly_unique_beneficiaries': 365,
                         'monthly_unique_claims': 373,
                         'monthly_unique_hcpcs': 275,
                         'provider_npi': 9979265126},
 'highest_payment_month': {'claim_month': '2009-03',
                           'monthly_claim_lines': 644,
                           'monthly_payment': 30830.0,
                           'monthly_unique_beneficiaries': 324,
                           'monthly_unique_claims': 332,
                           'monthly_unique_hcpcs': 270,
                           'provider_npi': 9979265126},
 'largest_payment_decrease': {'from_month': '2010-06',
                              'payment_change': -11210.0,
          

In [ ]:
print("Result keys:", result.keys())
print("Graph nodes:", investigation_graph.get_graph().nodes.keys())

Result keys: dict_keys(['provider_npi', 'profile', 'monthly_activity', 'population_comparison', 'hcpcs_activity', 'monthly_trends', 'report'])
Graph nodes: dict_keys(['__start__', 'collect_evidence', 'analyze_trends', 'build_report', '__end__'])


In [ ]:
print(result["report"].keys())
print(result["report"].get("monthly_trend_analysis"))

dict_keys(['provider_npi', 'screening', 'population_context', 'activity_summary', 'monthly_trend_analysis', 'hcpcs_summary', 'limitations'])
{'months_observed': 36, 'average_monthly_payment': 22713.333333333332, 'average_monthly_claim_lines': 505.6111111111111, 'highest_payment_month': {'provider_npi': 9979265126, 'claim_month': '2009-03', 'monthly_claim_lines': 644, 'monthly_unique_claims': 332, 'monthly_unique_beneficiaries': 324, 'monthly_payment': 30830.0, 'monthly_unique_hcpcs': 270}, 'highest_claim_month': {'provider_npi': 9979265126, 'claim_month': '2008-05', 'monthly_claim_lines': 700, 'monthly_unique_claims': 373, 'monthly_unique_beneficiaries': 365, 'monthly_payment': 28510.0, 'monthly_unique_hcpcs': 275}, 'largest_payment_increase': {'from_month': '2008-03', 'to_month': '2008-04', 'payment_change': 8800.0, 'payment_pct_change': 0.45595854922279794}, 'largest_payment_decrease': {'from_month': '2010-06', 'to_month': '2010-07', 'payment_change': -11210.0, 'payment_pct_change': 

In [ ]:
#investigator facing report 
def format_investigation_brief(report):
    provider = report["provider_npi"]
    screening = report["screening"]
    population = report["population_context"]
    activity = report["activity_summary"]
    trends = report["monthly_trend_analysis"]
    hcpcs = report["hcpcs_summary"]
    limitations = report["limitations"]

    lines = []

    lines.append("# Provider Investigation Brief")
    lines.append(f"\n**Provider NPI:** {provider}")

    # Screening
    lines.append("\n## 1. Screening")
    lines.append(screening.get("interpretation", "No interpretation available."))
    lines.append(
        f"\n- Anomaly prediction: {screening.get('anomaly_prediction')}"
        f"\n- Anomaly score: {screening.get('anomaly_score'):.4f}"
    )

    # Population context
    lines.append("\n## 2. Population Context")

    elevated = population.get("features_above_p99", [])
    grouped = group_elevated_features(elevated)

    if grouped:
        lines.append(
            f"{len(elevated)} features exceeded their population "
            "99th-percentile reference values, across these categories:"
        )

        for group_name, features in grouped.items():
            lines.append(f"\n### {group_name}")

            for item in features:
                lines.append(
                    f"- **{item['feature']}**: "
                    f"provider = {item['provider_value']:.3f}; "
                    f"population p99 = {item['population_p99']:.3f}"
                )
    else:
        lines.append("No features above the population p99 were reported.")

    # Activity
    lines.append("\n## 3. Activity Summary")
    lines.append(f"- Months observed: {activity['months_observed']}")
    lines.append(f"- Total claim lines: {activity['total_claim_lines']:,}")
    lines.append(f"- Total payments: ${activity['total_payment']:,.2f}")

    # Trends
    lines.append("\n## 4. Monthly Trend Analysis")
    lines.append(
        f"- Average monthly payment: "
        f"${trends['average_monthly_payment']:,.2f}"
    )
    lines.append(
        f"- Average monthly claim lines: "
        f"{trends['average_monthly_claim_lines']:,.1f}"
    )

    high_payment = trends["highest_payment_month"]
    high_claims = trends["highest_claim_month"]
    increase = trends["largest_payment_increase"]
    decrease = trends["largest_payment_decrease"]

    lines.append(
        f"- Highest payment month: {high_payment['claim_month']} "
        f"(${high_payment['monthly_payment']:,.2f})"
    )
    lines.append(
        f"- Highest claim-volume month: {high_claims['claim_month']} "
        f"({high_claims['monthly_claim_lines']:,} claim lines)"
    )
    lines.append(
        f"- Largest payment increase: "
        f"${increase['payment_change']:,.2f} "
        f"({increase['payment_pct_change']:.1%}) "
        f"from {increase['from_month']} to {increase['to_month']}"
    )
    lines.append(
        f"- Largest payment decrease: "
        f"${decrease['payment_change']:,.2f} "
        f"({decrease['payment_pct_change']:.1%}) "
        f"from {decrease['from_month']} to {decrease['to_month']}"
    )

    # HCPCS
    lines.append("\n## 5. HCPCS Activity")
    lines.append(
        f"The ten listed codes account for "
        f"{hcpcs['top_codes_payment_share']:.1%} of total payments."
    )
    lines.append("\n| HCPCS | Claim lines | Payment | Payment share |")
    lines.append("|---|---:|---:|---:|")

    for item in hcpcs.get("top_codes_by_payment", []):
        lines.append(
            f"| {item['hcpcs_code']} "
            f"| {item['claim_lines']:,} "
            f"| ${item['total_payment']:,.2f} "
            f"| {item['payment_share']:.1%} |"
        )

    # Limitations
    lines.append("\n## 6. Limitations")
    for limitation in limitations:
        lines.append(f"- {limitation}")

    lines.append(
        "\n## 7. Suggested Follow-up\n"
        "Review the elevated utilization and payment measures in context, "
        "including provider specialty, peer-group composition, and the "
        "underlying claims. Examine the months with the largest payment "
        "changes and assess whether the procedure mix helps explain them. "
        "The screening results alone do not establish fraud."
    )

    return "\n".join(lines)

In [ ]:
#run report
brief = format_investigation_brief(result["report"])
print(brief)

# Provider Investigation Brief

**Provider NPI:** 9979265126

## 1. Screening
Flagged by the Isolation Forest for further review.

- Anomaly prediction: -1
- Anomaly score: -0.1701

## 2. Population Context
11 features exceeded their population 99th-percentile reference values, across these categories:

### Utilization and volume
- **log_total_claim_lines**: provider = 9.809; population p99 = 3.912
- **log_unique_claims**: provider = 9.188; population p99 = 3.258
- **log_unique_beneficiaries**: provider = 8.796; population p99 = 3.045
- **log_unique_hcpcs**: provider = 7.361; population p99 = 3.638
- **log_avg_monthly_claim_lines**: provider = 6.228; population p99 = 2.079
- **log_max_monthly_claim_lines**: provider = 6.553; population p99 = 2.639

### Payments
- **log_total_payment**: provider = 13.614; population p99 = 7.779
- **log_avg_monthly_payment**: provider = 10.031; population p99 = 6.230

### Duration
- **active_months**: provider = 36.000; population p99 = 18.000

### Varia

In [ ]:
def group_elevated_features(elevated):
    groups = {
        "Utilization and volume": [
            "log_total_claim_lines",
            "log_unique_claims",
            "log_unique_beneficiaries",
            "log_unique_hcpcs",
            "log_avg_monthly_claim_lines",
            "log_max_monthly_claim_lines",
        ],
        "Payments": [
            "log_total_payment",
            "log_avg_monthly_payment",
        ],
        "Duration": [
            "active_months",
        ],
        "Variability": [
            "std_monthly_claim_lines",
            "std_monthly_payment",
        ],
    }

    grouped = {}

    for group_name, feature_names in groups.items():
        matches = [
            item for item in elevated
            if item["feature"] in feature_names
        ]

        if matches:
            grouped[group_name] = matches

    return grouped

In [ ]:
elevated = result["report"]["population_context"]["features_above_p99"]

grouped = group_elevated_features(elevated)

for group, features in grouped.items():
    print(f"\n{group}")
    for item in features:
        print(f"  - {item['feature']}")


Utilization and volume
  - log_total_claim_lines
  - log_unique_claims
  - log_unique_beneficiaries
  - log_unique_hcpcs
  - log_avg_monthly_claim_lines
  - log_max_monthly_claim_lines

Payments
  - log_total_payment
  - log_avg_monthly_payment

Duration
  - active_months

Variability
  - std_monthly_claim_lines
  - std_monthly_payment


26/10/01 20:55:47 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 904612 ms exceeds timeout 120000 ms
26/10/01 20:55:47 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/01 20:55:51 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [37]:
#analyze monthly trends: number of months above providers average numbers claim volume
#The percentage of total claim lines contributed by the five busiest months. 
#Whether the highest-volume months also had unusually high payments.
def analyze_monthly_consistency(state):
    monthly = state["monthly_activity"]

    claim_values = [
        row["monthly_claim_lines"] for row in monthly
    ]
    payment_values = [
        row["monthly_payment"] for row in monthly
    ]

    avg_claims = sum(claim_values) / len(claim_values)
    total_claims = sum(claim_values)
    total_payment = sum(payment_values)

    above_average_months = sum(
        value > avg_claims for value in claim_values
    )

    busiest_months = sorted(
        monthly,
        key=lambda row: row["monthly_claim_lines"],
        reverse=True
    )[:5]

    top_five_claims = sum(
        row["monthly_claim_lines"] for row in busiest_months
    )

    top_five_share = top_five_claims / total_claims

    return {
        "above_average_months": above_average_months,
        "months_observed": len(monthly),
        "top_five_months": busiest_months,
        "top_five_claim_share": top_five_share,
        "total_claim_lines": total_claims,
        "average_monthly_claim_lines": avg_claims,
        "total_monthly_payment": total_payment,
    }

In [ ]:
#test monthly trends
consistency = analyze_monthly_consistency({
    "monthly_activity": result["monthly_activity"]
})

print(consistency)

{'above_average_months': 23, 'months_observed': 36, 'top_five_months': [{'provider_npi': 9979265126, 'claim_month': '2008-05', 'monthly_claim_lines': 700, 'monthly_unique_claims': 373, 'monthly_unique_beneficiaries': 365, 'monthly_payment': 28510.0, 'monthly_unique_hcpcs': 275}, {'provider_npi': 9979265126, 'claim_month': '2008-10', 'monthly_claim_lines': 672, 'monthly_unique_claims': 356, 'monthly_unique_beneficiaries': 350, 'monthly_payment': 27110.0, 'monthly_unique_hcpcs': 282}, {'provider_npi': 9979265126, 'claim_month': '2009-07', 'monthly_claim_lines': 672, 'monthly_unique_claims': 344, 'monthly_unique_beneficiaries': 341, 'monthly_payment': 29820.0, 'monthly_unique_hcpcs': 288}, {'provider_npi': 9979265126, 'claim_month': '2008-09', 'monthly_claim_lines': 666, 'monthly_unique_claims': 338, 'monthly_unique_beneficiaries': 334, 'monthly_payment': 29080.0, 'monthly_unique_hcpcs': 279}, {'provider_npi': 9979265126, 'claim_month': '2008-07', 'monthly_claim_lines': 659, 'monthly_uniq

In [ ]:
pprint(result["report"])

Pretty printing has been turned ON


In [43]:
print(result["report"].keys())

dict_keys(['provider_npi', 'screening', 'population_context', 'activity_summary', 'monthly_trend_analysis', 'hcpcs_summary', 'monthly_consistency', 'limitations'])


In [44]:
for key, value in result["report"].items():
    print(key, "→", "EMPTY" if not value else "OK")

provider_npi → OK
screening → OK
population_context → OK
activity_summary → OK
monthly_trend_analysis → OK
hcpcs_summary → OK
monthly_consistency → OK
limitations → OK


26/10/05 12:02:00 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 907114 ms exceeds timeout 120000 ms
26/10/05 12:02:00 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/05 12:02:01 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o